Cuaderno usado para el entrenamiento del modelo 2 de probabilidades de zonas censales

# librerias y datos

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.0 MB/s eta 0:00:00


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

import pandas as pd

In [ ]:
codigos_comunas_fuera_muestra = [
    15202,  # General Lagos
    1403,  # Colchane
    2202,  # Ollagüe
    5104,  # Juan Fernández
    5201,  # Isla de Pascua
    10103,  # Cochamó
    10401,  # Chaitén
    10402,  # Futaleufú
    10403,  # Hualaihué
    10404,  # Palena
    11102,  # Lago Verde
    11203,  # Guaitecas
    11302,  # O’Higgins
    11303,  # Tortel
    12102,  # Laguna Blanca
    12103,  # Río Verde
    12104,  # San Gregorio
    12201,  # Cabo de Hornos (Ex Navarino)
    12202,  # Antártica
    12302,  # Primavera
    12303,  # Timaukel
    12402   # Torres del Paine
]


In [ ]:
dataset = pd.read_csv('/content/drive/MyDrive/censo_hogares_final_final_final_2.csv')
dataset['zona_censal'] = dataset['row'].apply(lambda x: x.split('_')[2])
dataset['zona_censal'] = dataset['zona_censal'].astype(str)
dataset['comuna'] = dataset['row'].apply(lambda x: x.split('_')[0])

# elimniar comunas fuera de la muestra
dataset = dataset[~dataset['comuna'].isin(codigos_comunas_fuera_muestra)]


In [ ]:
dataset["COMUNA"]

,COMUNA
0,13119
1,13119
2,13119
3,13119
4,13119
...,...
4795993,16304
4795994,16304
4795995,16304
4795996,16304


# modelo de prediccion aleatorio


**Accuracy del método aleatorio**

Como línea base, consideramos un clasificador que asigna cada hogar a una zona censal al azar, respetando la distribución empírica de frecuencias. Sea $(\pi_j)$ la proporción de hogares que pertenecen a la zona censal $(j)$ en el conjunto de datos. Si asignamos aleatoriamente cada hogar a la zona $(j)$ con probabilidad $(\pi_j)$, la probabilidad de acierto (accuracy) para un hogar dado es:

$$
\Pr(\text{correcto})
= \sum_{j=1}^{J} \pi_j \times \Pr(\text{predice } j \mid \text{real } j)
= \sum_{j=1}^{J} \pi_j \times \pi_j
= \sum_{j=1}^{J} \pi_j^2
$$

Por lo tanto, el accuracy esperado de este clasificador aleatorio—que asigna zonas con probabilidades $\pi_j$ viene dado por:

$$
\mathrm{Accuracy}_{\mathrm{random}}
= \sum_{j=1}^{J} \pi_j^2
$$

Este valor proporciona un umbral mínimo de referencia: cualquier modelo supervisado debe superar esta base aleatoria para considerarse informativo.


In [ ]:
import pandas as pd

def baseline_random_por_comuna(dataset, comuna_code):
    """
    Calcula la baseline de clasificación aleatoria por comuna,
    es decir, la suma de π_i^2 donde π_i es la proporción de hogares
    en la zona i dentro de la comuna.

    Parámetros:
    - dataset: DataFrame con columnas 'comuna' y 'zona_censal'.
    - comuna_code: código de comuna para filtrar.

    Retorna:
    - random_accuracy (float): ∑(π_i^2)
    """
    # Filtrar por comuna
    df_com = dataset[dataset['comuna'] == comuna_code]
    if df_com.empty:
        raise ValueError(f"No hay datos para la comuna {comuna_code}")

    # Frecuencias relativas de 'zona_censal'
    freqs = df_com['zona_censal'].value_counts(normalize=True)

    # Baseline aleatoria: suma de π_i^2
    random_acc = (freqs ** 2).sum()
    return random_acc


In [ ]:

# Obten la lista de códigos únicos de comuna
comunas = dataset['comuna'].unique().tolist()

# Diccionario para almacenar la baseline aleatoria
resultados = {}

for comuna_code in comunas:
    try:
        rand_acc = baseline_random_por_comuna(dataset, comuna_code)
        resultados[comuna_code] = rand_acc
    except ValueError:
        resultados[comuna_code] = None

# Convertir a DataFrame para mejor visualización
df_random = pd.DataFrame.from_dict(
    resultados, orient='index', columns=['random_accuracy']
).reset_index().rename(columns={'index':'comuna'})

print(df_random)

    comuna  random_accuracy
0    13119         0.101064
1    10302         0.088369
2     5303         0.707318
3    13110         0.168278
4     8301         0.086630
..     ...              ...
341   2302         0.506881
342  12102         0.238015
343  12202         1.000000
344   5104         0.861970
345   2202         0.589286

[346 rows x 2 columns]


# modelo de prediccion random forest

**Accuracy del Random Forest por comuna**

Para evaluar el desempeño de un modelo Random Forest a nivel de comuna, seguimos estos pasos:

1. **Filtrado por comuna**  
   Seleccionamos únicamente los registros cuyo código de comuna coincida con el valor `comuna_code`.  

2. **Codificación de la variable objetivo**  
   Convertimos la zona censal (variable categórica) en etiquetas numéricas con `LabelEncoder`, obteniendo `zona_censal_enc`.  

3. **Selección de características**  
   Definimos el conjunto de predictores \(X\) con las siguientes columnas:  
   - `educacion_jefe_hogar`  
   - `hogar_urbano`  
   - `mujer_jefe_hogar`  
   - `densidad_hogar`  
   - `educacion_promedio_hogar`  
   - `jefe_trabajo_semana_pasada`  
   - `material_aceptable`  
   - `material_recuperable`  
   - `dependencia_hogar`  
   - `numero_personas`  
   - `edad_promedio_hogar`  

   Y la variable objetivo \(y\) es `zona_censal_enc`.  


4. **Entrenamiento del Random Forest**  
   Creamos un `RandomForestClassifier` con:
   - `n_estimators=100` (número de árboles).
   - `class_weight='balanced'` para ajustar automáticamente los pesos de clases desbalanceadas.
   - `random_state=42` para reproducibilidad.
   - `n_jobs=-1` para usar todos los núcleos disponibles.

   Ajustamos (`fit`) el modelo sobre $(X_{\mathrm{train}}, y_{\mathrm{train}})$.

5. **Cálculo de accuracy**  
   - Predecimos las etiquetas en el conjunto de test: `y_pred = rf.predict(X_test)`.  
   - Calculamos el accuracy con `accuracy_score(y_test, y_pred)`, que mide la proporción de hogares correctamente asignados a su zona censal.



In [ ]:

def entrenar_rf_por_comuna(dataset, comuna_code, test_size=0.2, random_state=42):
    """
    Filtra el dataset por código de comuna, entrena un Random Forest y devuelve el accuracy.

    Parámetros:
    - dataset: DataFrame original con la columna 'comuna' y características.
    - comuna_code: valor de la comuna por la que filtrar.
    - test_size: proporción de datos para test.
    - random_state: semilla para reproducibilidad.

    Retorna:
    - accuracy (float)
    """
    # Filtrar dataset
    data_filtrada = dataset[dataset['comuna'] == comuna_code].copy()

    # Encoder de la variable objetivo
    le = LabelEncoder()
    data_filtrada['zona_censal_enc'] = le.fit_transform(data_filtrada['zona_censal'])

    # Selección de features y target
    X = data_filtrada[[
    'educacion_jefe_hogar', 'hogar_urbano', 'mujer_jefe_hogar',
    'densidad_hogar', 'educacion_promedio_hogar', 'jefe_trabajo_semana_pasada',
    'material_aceptable', 'material_recuperable', 'dependencia_hogar',
    'numero_personas', 'edad_promedio_hogar',
    'fraccion_ancianos',
    'numero_ancianos', 'numero_habitaciones', 'fraccion_niños',
    'numero_niños', 'techo_zinc', 'paredes_madera', 'piso_madera',
    'piso_radier', 'paredes_ladrillo', 'techo_paja'
    ]]

    y = data_filtrada['zona_censal_enc']

    # Train-test split estratificado
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # Entrenar Random Forest con balanceo de clases
    rf = RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=random_state,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    # Predicción y cálculo de accuracy
    y_pred = rf.predict(X_test)
    return accuracy_score(y_test, y_pred)

In [ ]:

# Obten la lista de códigos únicos de comuna
comunas = dataset['comuna'].unique().tolist()

# Diccionario para almacenar resultados
resultados = {}

for comuna_code in comunas:
    try:
        acc = entrenar_rf_por_comuna(dataset, comuna_code)
        resultados[comuna_code] = acc
    except ValueError:
        resultados[comuna_code] = None

# Convertir a DataFrame para mejor visualización
df_resultados = pd.DataFrame.from_dict(
    resultados, orient='index', columns=['accuracy']
).reset_index().rename(columns={'index':'comuna'})



print(df_resultados)

    comuna  accuracy
0    13119  0.125482
1    10302  0.322709
2     5303  0.854779
3    13110  0.188467
4     8301  0.155483
..     ...       ...
341   2302  0.707031
342  12102  0.421053
343  12202  1.000000
344   5104  0.933333
345   2202  0.941176

[346 rows x 2 columns]


# modelo de prediccion random forest(guardar)


In [ ]:
from pathlib import Path
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

def entrenar_rf_por_comuna(
    dataset,
    comuna_code,
    test_size=0.2,
    random_state=42,
    modelos_dir="/content/drive/MyDrive/modelos_2_rf"
):
    """
    Filtra el dataset por código de comuna, entrena un Random Forest y
    devuelve accuracy + ruta al archivo del modelo guardado.
    El modelo se guarda como <modelos_dir>/rf_<comuna_code>.joblib
    """
    # 1. Filtrar la comuna
    data = dataset[dataset["comuna"] == comuna_code].copy()
    if data.empty:
        raise ValueError(f"No hay datos para la comuna {comuna_code}")

    # 2. Codificar etiqueta
    data["zona_censal_enc"] = LabelEncoder().fit_transform(data["zona_censal"])

    # 3. Seleccionar X, y
    base_cols = [
        'educacion_jefe_hogar', 'hogar_urbano', 'mujer_jefe_hogar',
        'densidad_hogar', 'educacion_promedio_hogar', 'jefe_trabajo_semana_pasada',
        'material_aceptable', 'material_recuperable', 'dependencia_hogar',
        'numero_personas', 'edad_promedio_hogar', 'fraccion_ancianos',
        'numero_ancianos', 'numero_habitaciones', 'fraccion_niños',
        'numero_niños', 'techo_zinc', 'paredes_madera', 'piso_madera',
        'piso_radier', 'paredes_ladrillo', 'techo_paja'
    ]
    X, y = data[base_cols], data["zona_censal_enc"]

    # 4. Hold-out (estratificado para estabilidad)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, stratify=y, test_size=test_size, random_state=random_state
    )

    # 5. Entrenar RF
    rf = RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=random_state,
        n_jobs=-1
    )
    rf.fit(X_tr, y_tr)

    # 6. Evaluar
    acc = accuracy_score(y_te, rf.predict(X_te))

    # 7. Guardar el modelo
    modelos_dir = Path(modelos_dir)
    modelos_dir.mkdir(parents=True, exist_ok=True)
    modelo_path = modelos_dir / f"rf_{comuna_code}.joblib"
    joblib.dump(rf, modelo_path)

    return acc, str(modelo_path)


In [ ]:
import pandas as pd
import numpy as np

# -----------------------------------------------------------
# 0. Lista de comunas
comunas = dataset["comuna"].unique().tolist()

# 1. Acumularemos filas como diccionarios
filas = []

for comuna_code in comunas:
    try:
        acc, path = entrenar_rf_por_comuna(dataset, comuna_code)
        filas.append({"comuna": comuna_code, "accuracy": acc, "model_path": path})
    except ValueError:               # por si alguna comuna no tiene datos válidos
        filas.append({"comuna": comuna_code, "accuracy": np.nan, "model_path": None})

# 2. DataFrame final
df_resultados = pd.DataFrame(filas).sort_values("comuna").reset_index(drop=True)

print(df_resultados.head())


  comuna  accuracy                                         model_path
0  10101  0.140510  /content/drive/MyDrive/modelos_2_rf/rf_10101.j...
1  10102  0.319319  /content/drive/MyDrive/modelos_2_rf/rf_10102.j...
2  10103  0.219608  /content/drive/MyDrive/modelos_2_rf/rf_10103.j...
3  10104  0.398915  /content/drive/MyDrive/modelos_2_rf/rf_10104.j...
4  10105  0.308630  /content/drive/MyDrive/modelos_2_rf/rf_10105.j...


In [ ]:
from pathlib import Path
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

def entrenar_rf_por_comuna(
    dataset,
    comuna_code,
    test_size: float = 0.20,
    random_state: int = 42,
    modelos_dir: str = "/content/drive/MyDrive/modelos_2_rf"
):
    """
    Entrena un Random Forest por comuna y guarda:
      • el modelo  -> rf_<comuna_code>.joblib
      • el encoder -> rf_<comuna_code>_encoder.joblib

    Returns
    -------
    acc : float
        Exactitud sobre el hold-out.
    model_path : str
        Ruta al archivo del modelo guardado.
    encoder_path : str
        Ruta al archivo del LabelEncoder guardado.
    """
    # 1. Sub-muestra de la comuna
    data = dataset[dataset["comuna"] == comuna_code].copy()
    if data.empty:
        raise ValueError(f"No hay datos para la comuna {comuna_code}")

    # 2. Codificar etiqueta
    le = LabelEncoder()
    data["zona_censal_enc"] = le.fit_transform(data["row"])

    # 3. Variables predictoras y objetivo
    base_cols = [
        'educacion_jefe_hogar', 'hogar_urbano', 'mujer_jefe_hogar',
        'densidad_hogar', 'educacion_promedio_hogar', 'jefe_trabajo_semana_pasada',
        'material_aceptable', 'material_recuperable', 'dependencia_hogar',
        'numero_personas', 'edad_promedio_hogar', 'fraccion_ancianos',
        'numero_ancianos', 'numero_habitaciones', 'fraccion_niños',
        'numero_niños', 'techo_zinc', 'paredes_madera', 'piso_madera',
        'piso_radier', 'paredes_ladrillo', 'techo_paja'
    ]
    X, y = data[base_cols], data["zona_censal_enc"]

    # 4. Hold-out estratificado
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, stratify=y, test_size=test_size, random_state=random_state
    )

    # 5. Entrenamiento
    rf = RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=random_state,
        n_jobs=-1
    )
    rf.fit(X_tr, y_tr)

    # 6. Evaluación
    acc = accuracy_score(y_te, rf.predict(X_te))

    # 7. Guardado (modelo + encoder)
    modelos_dir = Path(modelos_dir)
    modelos_dir.mkdir(parents=True, exist_ok=True)

    model_path   = modelos_dir / f"rf_{comuna_code}.joblib"
    encoder_path = modelos_dir / f"rf_{comuna_code}_encoder.joblib"

    joblib.dump(rf, model_path)
    joblib.dump(le, encoder_path)

    return acc, str(model_path), str(encoder_path)


In [ ]:
import pandas as pd
import numpy as np

comunas = dataset["comuna"].unique().tolist()

filas = []

for comuna_code in comunas:
    try:
        acc, m_path, e_path = entrenar_rf_por_comuna(dataset, comuna_code)
        filas.append({
            "comuna": comuna_code,
            "accuracy": acc,
            "model_path": m_path,
            "encoder_path": e_path
        })
    except ValueError:  # comuna sin datos
        filas.append({
            "comuna": comuna_code,
            "accuracy": np.nan,
            "model_path": None,
            "encoder_path": None
        })

df_resultados = (
    pd.DataFrame(filas)
      .sort_values("comuna")
      .reset_index(drop=True)
)

print(df_resultados.head())


  comuna  accuracy                                         model_path  \
0  10101  0.137690  /content/drive/MyDrive/modelos_2_rf/rf_10101.j...   
1  10102  0.320915  /content/drive/MyDrive/modelos_2_rf/rf_10102.j...   
2  10103  0.219608  /content/drive/MyDrive/modelos_2_rf/rf_10103.j...   
3  10104  0.404342  /content/drive/MyDrive/modelos_2_rf/rf_10104.j...   
4  10105  0.287054  /content/drive/MyDrive/modelos_2_rf/rf_10105.j...   

                                        encoder_path  
0  /content/drive/MyDrive/modelos_2_rf/rf_10101_e...  
1  /content/drive/MyDrive/modelos_2_rf/rf_10102_e...  
2  /content/drive/MyDrive/modelos_2_rf/rf_10103_e...  
3  /content/drive/MyDrive/modelos_2_rf/rf_10104_e...  
4  /content/drive/MyDrive/modelos_2_rf/rf_10105_e...  


#modelo de prediccion catboost (test)

**Accuracy del CatBoost por comuna**

CatBoost es un algoritmo de boosting basado en árboles de decisión que mejora la precisión y reduce el sobreajuste mediante dos innovaciones clave: el manejo nativo de variables categóricas sin necesidad de one‑hot encoding y el uso de “ordered boosting” para calcular estadísticas de forma aleatoria, minimizando el sesgo en cada iteración.

Para evaluar el desempeño de un modelo CatBoost a nivel de comuna, seguimos estos pasos:

1. **Filtrado por comuna**  
   Seleccionamos únicamente los registros cuyo código de comuna coincida con el valor `comuna_code`.  

2. **Codificación de la variable objetivo**  
   Convertimos la zona censal (variable categórica) en etiquetas numéricas con `LabelEncoder`, obteniendo `zona_enc`.  

3. **Selección de características**  
   Definimos el conjunto de predictores \(X\) con las siguientes columnas:  
   - `educacion_jefe_hogar`  
   - `hogar_urbano`  
   - `mujer_jefe_hogar`  
   - `densidad_hogar`  
   - `educacion_promedio_hogar`  
   - `jefe_trabajo_semana_pasada`  
   - `material_aceptable`  
   - `material_recuperable`  
   - `dependencia_hogar`  
   - `numero_personas`  
   - `edad_promedio_hogar`  

   Y la variable objetivo \(y\) es `zona_enc`.  

4. **Entrenamiento del CatBoostClassifier**  
   Creamos un `CatBoostClassifier` con:  
   - `iterations=500` (número de rondas de boosting).  
   - `learning_rate=0.1` (tasa de aprendizaje).  
   - `depth=6` (profundidad máxima de cada árbol).  
   - `random_seed=42` para reproducibilidad.  
   - `verbose=False` para suprimir la salida de entrenamiento.  
   - `cat_features=cat_features` para indicar al modelo las columnas categóricas.  

   Ajustamos (`fit`) el modelo sobre $(X_{\mathrm{train}}, y_{\mathrm{train}})$.

5. **Cálculo de accuracy**  
   - Predecimos las etiquetas en el conjunto de test:  
     ```python
     y_pred = model.predict(X_test)
     ```  
   - Calculamos el accuracy con:  
     ```python
     accuracy_score(y_test, y_pred)
     ```

   El resultado es la proporción de hogares de la comuna correctamente asignados a su zona censal por el modelo CatBoost.  




In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

def entrenar_cb_por_comuna(dataset, comuna_code, cat_features=None,
                           test_size=0.2, random_state=42,
                           iterations=500, learning_rate=0.1, depth=6):
    """
    Filtra el dataset por código de comuna, entrena un CatBoostClassifier y devuelve el accuracy.

    Parámetros:
    - dataset: DataFrame con columna 'comuna', 'zona_censal' y características.
    - comuna_code: valor de la comuna por la que filtrar.
    - cat_features: lista de nombres de columnas categóricas.
    - test_size: proporción de datos para test.
    - random_state: semilla para reproducibilidad.
    - iterations, learning_rate, depth: hiperparámetros de CatBoost.

    Retorna:
    - accuracy (float)
    """
    # 1. Filtrar dataset
    df = dataset[dataset['comuna'] == comuna_code].copy()

    # 2. Codificar target
    le = LabelEncoder()
    df['zona_enc'] = le.fit_transform(df['zona_censal'])

    # 3. Dividir features/target
    features = [
    'educacion_jefe_hogar', 'hogar_urbano', 'mujer_jefe_hogar',
    'densidad_hogar', 'educacion_promedio_hogar', 'jefe_trabajo_semana_pasada',
    'material_aceptable', 'material_recuperable', 'dependencia_hogar',
    'numero_personas', 'edad_promedio_hogar',
    'fraccion_ancianos',
    'numero_ancianos', 'numero_habitaciones', 'fraccion_niños',
    'numero_niños', 'techo_zinc', 'paredes_madera', 'piso_madera',
    'piso_radier', 'paredes_ladrillo', 'techo_paja'
    ]
    X = df[features]
    y = df['zona_enc']

    # 4. Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # 5. Crear y entrenar CatBoost
    model = CatBoostClassifier(
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        random_seed=random_state,
        verbose=False
    )
    model.fit(X_train, y_train, cat_features=cat_features)

    # 6. Predecir y calcular accuracy
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)


In [ ]:

# Obtenemos la lista de códigos únicos de comuna
comunas = dataset['comuna'].unique().tolist()

# Diccionario para almacenar resultados
resultados_cat = {}

for comuna_code in comunas:
    try:
        acc = entrenar_cb_por_comuna(dataset, comuna_code)
        resultados_cat[comuna_code] = acc
    except Exception:
        resultados_cat[comuna_code] = None

# Convertir a DataFrame para mejor visualización
df_resultados_cat = pd.DataFrame.from_dict(
    resultados_cat, orient='index', columns=['accuracy']
).reset_index().rename(columns={'index':'comuna'})



print(df_resultados_cat)


    comuna  accuracy
0    13119  0.172215
1    10302  0.380478
2     5303  0.860294
3    13110  0.223107
4     8301  0.197943
..     ...       ...
341   2302  0.683594
342  12102  0.473684
343  12202       NaN
344   5104  0.916667
345   2202  0.882353

[346 rows x 2 columns]


In [ ]:
df_resultados_cat.to_csv('/content/drive/MyDrive/df_resultados_cat.csv', index=False)

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/df_resultados_cat.csv')

In [ ]:
dataset['comuna'].unique().tolist()

In [ ]:
import pandas as pd

df_resultados_cat = df
# Asegurarse de que la columna 'comuna' sea del mismo tipo en todos los dataframes
df_random['comuna'] = df_random['comuna'].astype(str)
df_resultados['comuna'] = df_resultados['comuna'].astype(str)
df_resultados_cat['comuna'] = df_resultados_cat['comuna'].astype(str)

# Realizar el primer merge entre df_random y df_resultados
# Usamos un outer merge para mantener todas las comunas y manejar posibles NaN si una comuna no está en un dataframe
df_merged_1 = pd.merge(df_random, df_resultados, on='comuna', how='outer', suffixes=('_random', '_rf'))

# Realizar el segundo merge con df_resultados_cat
df_merged_final = pd.merge(df_merged_1, df_resultados_cat, on='comuna', how='outer', suffixes=('_rf_merged', '_catboost'))

In [ ]:
# Renombrar la columna 'accuracy' de df_resultados_cat para evitar conflictos
df_merged_final.rename(columns={'accuracy': 'accuracy_catboost'}, inplace=True)

# Calcular las diferencias de accuracy
df_merged_final['diff_rf_vs_random'] = df_merged_final['accuracy_rf_merged'] - df_merged_final['random_accuracy']
df_merged_final['diff_catboost_vs_random'] = df_merged_final['accuracy_catboost'] - df_merged_final['random_accuracy']
df_merged_final['diff_rf_vs_catboost'] = df_merged_final['accuracy_rf_merged'] - df_merged_final['accuracy_catboost']

In [ ]:
df_merged_final['diff_rf_vs_catboost']

,diff_rf_vs_catboost
0,-0.051607
1,-0.039383
2,-0.035294
3,-0.032564
4,-0.033771
...,...
341,-0.013208
342,-0.016345
343,-0.027491
344,0.000959
